# 03 · Delay model diagnostics

Test months only (never used for training).

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(ROOT / "backend"))
import numpy as np, pandas as pd
pd.set_option("display.width", 140)

In [2]:
import json
current = json.loads((ROOT / "ml/artifacts/current.json").read_text())["version"]
meta = json.loads((ROOT / f"ml/artifacts/delay_model_{current}/meta.json").read_text(encoding="utf-8"))
print("train", meta["train_months"], "test", meta["test_months"])
pd.DataFrame({k: meta["metrics"][k] for k in ["model_plan", "model_live", "baseline_hist_quantiles",
                                              "baseline_hist_median", "coverage_plan"]}).round(3)

train ['2026-03', '2026-04', '2026-05', '2026-06'] test ['2026-07', '2026-08']


,model_plan,model_live,baseline_hist_quantiles,baseline_hist_median,coverage_plan
0.05,0.199,0.199,0.196,0.632,0.309
0.10,0.384,0.384,0.383,0.745,0.350
0.25,0.922,0.914,0.922,1.084,0.253
0.50,1.595,1.569,1.650,1.650,0.493
0.75,1.819,1.773,1.932,2.215,0.732
0.90,1.432,1.391,1.536,2.555,0.885
0.95,1.055,1.025,1.127,2.668,0.939
0.98,0.645,0.631,0.679,2.736,0.973


Coverage below the 0.5 quantile is far above nominal because delays are whole minutes and most trains are 0–1 min late: `P(delay ≤ q)` includes a big point mass. Upper quantiles, which drive missed connections, are close to nominal.

In [3]:
pd.Series(meta["feature_importance"]).sort_values(ascending=False).round(0)

eva             8738.0
line            5230.0
train_type      2637.0
station_num     1360.0
prev_delay      1298.0
hour            1176.0
weekday          417.0
position         146.0
is_departure      19.0
product            8.0
dtype: float64

In [4]:
ev = json.loads((ROOT / "ml/artifacts/evaluation.json").read_text(encoding="utf-8"))
pd.DataFrame(ev["brier"]).T.round(3)

,observed_rate,timetable_only,historical_median,simulator
held,0.759,0.241,0.168,0.127
on_time,0.527,0.473,0.391,0.216
stranded,0.043,0.043,0.043,0.033


In [5]:
pd.DataFrame(ev["by_buffer"]).round(3)

,buffer,journeys,observed_held,predicted_held,brier_simulator,brier_timetable
0,≤ 5 min,93,0.269,0.375,0.200,0.731
1,6-10 min,187,0.588,0.659,0.188,0.412
2,> 10 min,240,0.796,0.864,0.162,0.204


In [6]:
pd.DataFrame(ev["by_pair"]).sort_values("gap", key=abs, ascending=False).head(8).round(3)

,pair,journeys,observed_on_time,predicted_on_time,gap
4,Stuttgart Hauptbahnhof (oben) -> Villingen Bah...,76,0.290,0.429,0.139
9,Freiburg Hauptbahnhof -> Tuttlingen Bahnhof,59,0.492,0.613,0.122
11,Tübingen Hauptbahnhof -> Villingen Bahnhof/ZOB,55,0.364,0.473,0.110
8,Karlsruhe Hauptbahnhof -> Tuttlingen Bahnhof,60,0.683,0.606,-0.077
10,Stuttgart Hauptbahnhof (oben) -> Donaueschinge...,55,0.418,0.488,0.070
1,Offenburg Bahnhof -> St. Georgen Bahnhof,100,0.660,0.595,-0.065
3,Singen (Htw) Bahnhof -> Rottweil Bahnhof,90,0.556,0.524,-0.031
0,Freiburg Hauptbahnhof -> Villingen Bahnhof/ZOB,104,0.740,0.769,0.029
